In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 8 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 calibration.
# - Centre search around the ACTUAL incumbent.
# - Use local + moderate-wide + global diagnostic pools.
# - Compare EI, posterior mean and UCB.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------

week10_pred_mean = 9.997490834716594
week10_pred_std = 0.018147866338162684
week10_actual = 9.9891379221651

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(200000, 8)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(150000, 8)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(120000, 8)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Domain-boundary check
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (50, 8)
Y shape: (50,)

Current best:
[0.117062 0.183135 0.104304 0.120414 0.770644 0.497652 0.179588 0.558852] -> 9.9940246476926

Y range:
min = 5.5921933895401965
max = 9.9940246476926
std = 1.2021466085892336

WEEK 10 CALIBRATION CHECK
Predicted mean: 9.997490834716594
Predicted std : 0.018147866338162684
Actual        : 9.9891379221651

Prediction error:
-0.008352912551494285

Error / predicted std:
-0.46026967555569653


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co


GP FIT

Fitted kernel:
0.868**2 * Matern(length_scale=[1.27, 2, 1, 2, 2, 2, 1.35, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[1.26704772 2.         1.00003349 2.         2.         2.
 1.34919161 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15689373 0.09939592 0.19878519 0.09939592 0.09939592 0.09939592
 0.14734145 0.09939592]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08 0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15 0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
469999

PRIMARY EI
candidate = [0.         0.16541153 0.11597226 0.07838899 0.77108674 0.41304155
 0.2386966  0.79401349]
mean = 9.964343396136725
std = 0.0804188913183294
EI = 0.01940257407537608

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.         0.16541153 0.11597226 0.07838899 0.77108674 0.41304155
 0.2386966  0.79401349] 
 mean = 9.964343 
 std = 0.080419 
 EI = 0.01940257 

xi = 1.202147e-02 
 candidate = [0.         0.         0.         0.09510

In [2]:
# ============================================================
# FINAL FUNCTION 8 - WEEK 11 SELECTION
# ============================================================
#
# Week 10 calibration remained good (~ -0.46 sigma).
#
# EI and high-beta UCB move much farther from the proven basin
# and are driven by increased uncertainty.
#
# The highest posterior mean stays relatively local, has low
# predictive uncertainty, and remains inside the domain.
#
# Therefore select the highest predicted mean candidate.
# ============================================================

final_idx = np.argmax(mu)

week11_candidate = candidates[final_idx]

print("================================")
print("FUNCTION 8 - WEEK 11 FINAL")
print("================================")

print("\nCandidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 8 - WEEK 11 FINAL

Candidate:
[0.10732281 0.14904063 0.12909354 0.15744106 0.81531325 0.49317471
 0.2015848  0.58604084]

Predicted mean:
9.998586911180592

Predicted std:
0.013699035186499755

Distance from current best:
0.08050641206048542

Portal format:
0.107323-0.149041-0.129094-0.157441-0.815313-0.493175-0.201585-0.586041
